# 1. Business Understanding and Analytic Approach

This notebook frames the study *Measuring the Real Value of Weather in Flight Delay Prediction before Departure*: the business problem, the research questions, the analytic approach and the data requirements. Data understanding, visual EDA and statistical tests follow in notebooks 2–4 of this folder; the modelling pipeline is in the numbered notebooks at the project root.

## 1.1 Business problem

Arrival delays impose costs on airlines (crew and aircraft rotations, missed connections), airports (gate and runway capacity) and passengers. Weather is one of the main causes of delay outside airline control, so delay-prediction models are commonly augmented with meteorological features.

The decision of interest is a **pre-departure prediction**: an airline operations centre, an airport or a passenger application issues a forecast for a flight **$H$ hours before its scheduled departure** $t^{\text{dep}}$, i.e., at issuance time $t^0 = t^{\text{dep}} - H$. Only information available at $t^0$ can be used by a deployed system.

**The problem.** Most weather-augmented models join the *realized* (observed) weather at the scheduled departure hour (origin) and the scheduled arrival hour (destination). At issuance time the destination observation lies hours in the future, and the origin observation may also post-date $t^0$. Such models are therefore evaluated with information no deployed system could have — a form of temporal leakage that is less visible than leakage from post-departure fields such as `DepDelay` or `TaxiOut`. As a result, the reported value of weather information may be overstated.

## 1.2 Research questions and expected contributions

| # | Question | Evidence in the paper |
|---|---|---|
| RQ1 | How much of the predictive value of weather survives when only information available at issuance time is used, for horizons $H \in \{0, 2, 6\}$ h? | Oracle vs. deployable comparison (Sect. 5.1) |
| RQ2 | How is the weather gain split between origin and destination, and are the two endpoints complementary or redundant? | Four-cell ablation with two-player Shapley attribution (Sect. 5.2) |
| RQ3 | Does weather still add value once the state of the previous leg of the same aircraft is known? | Horizon-aware delay-propagation features (Sect. 5.3) |
| RQ4 | Which alignment errors in the common BTS–NOAA join affect the validity of weather features? | Pipeline audit (Sect. 3.2, 4.2) |

Robustness of the conclusions across model families is checked in Sect. 5.4.

**Headline results (test set May–July 2022).**
- Deployable weather retains 79%, 48% and 24% of the oracle $R^2$ gain at $H = 0$, 2 and 6 h (85%, 66% and 45% of the ROC-AUC gain).
- Origin and destination contribute comparably (origin share 49–58%) and partly redundantly (negative interaction).
- Previous-leg features raise $R^2$ from 0.02 to 0.30 at $H = 0$ (0.32 with weather); weather keeps 45–72% of its $R^2$ gain on top of them, more at longer horizons.
- Four classes of alignment errors were corrected, e.g., 2.63% of flights had destination weather shifted by 24 h.

## 1.3 Success criteria and metrics

| Aspect | Choice | Rationale |
|---|---|---|
| Targets | `ArrDelayMinutes` = max(0, `ArrDelay`) (regression); `ArrDel15` (classification) | Standard BTS delay definitions |
| Primary metrics | $R^2$, ROC-AUC, PR-AUC | 67% of training flights have zero delay, so the MAE-optimal constant is zero and MAE discriminates weakly between models |
| Secondary metrics | MAE, RMSE, $F_1$ (threshold tuned on validation), Brier score | Reported together with constant baselines |
| Value of weather | Gain over the schedule-only model; retention = deployable gain / oracle gain | Directly answers RQ1 |
| Uncertainty | Paired day-level block bootstrap (1,000 resamples) for ΔMAE and ΔBrier; three seeds for the main results | Flights on the same day share weather and congestion |

## 2. Analytic Approach

**Problem type.** Supervised learning on tabular data: regression of arrival delay minutes and binary classification of delays ≥ 15 min, evaluated on a chronological hold-out.

**Information settings.** Weather is aggregated into hourly buckets $[b, b+1\,\text{h})$; a bucket is fully observed only at $b + 1$ h.

| Setting | Origin weather | Destination weather | Deployable |
|---|---|---|---|
| Oracle | bucket containing $t^{\text{dep}}$ | bucket containing $t^{\text{arr}}$ | No (upper bound) |
| $H \in \{0, 2, 6\}$ h | last closed bucket, $b = \lfloor t^0 \rfloor_{\text{h}} - 1$ h | same bucket, at the destination | Yes |

Deployable settings add backward-looking 3-hour summaries (e.g., thunder in the last 3 h, minimum visibility, pressure tendency) and the lead time $t^{\text{arr}} - t^0$.

**Experimental design.**
1. *Ablation and attribution:* four weather configurations per setting (none, origin, destination, both); the gain is split between endpoints with the two-player Shapley value $\phi_{\text{org}} = \tfrac12[(v_{\text{org}} - v_\emptyset) + (v_{\text{both}} - v_{\text{dst}})]$; the interaction $v_{\text{both}} - v_{\text{org}} - v_{\text{dst}} + v_\emptyset$ indicates redundancy ($<0$) or synergy ($>0$).
2. *Delay propagation:* six features of the previous leg of the same aircraft, encoded with only the information realized at $t^0$ (status, departure/arrival delay if realized, overdue minutes, scheduled turnaround, turnaround slack).
3. *Models:* LightGBM as backbone (squared loss / log loss, early stopping on validation); XGBoost, random forest, linear/logistic regression and L1/Huber objectives as robustness checks.
4. *Validation protocol:* chronological split; out-of-fold target encoding of high-cardinality identifiers; all encoders and caps fit on the training period only.

**Workflow and notebooks.**

| Step | Notebook | Purpose |
|---|---|---|
| 1 | `01_download_noaa_lcd.ipynb` | Download NOAA LCD hourly reports |
| 2 | `02_weather_processing.ipynb` | Audited weather cleaning and UTC alignment |
| 3 | `03_flights_information_settings.ipynb` | Flight cleaning, UTC times, oracle and deployable weather sets |
| 4 | `04_eda_paper_statistics.ipynb` and `EDA/` | Exploratory analysis (training period) |
| 5 | `05_modeling_horizon.ipynb` | Main experiment (RQ1, RQ2) |
| 6 | `06_propagation.ipynb` | Weather given delay propagation (RQ3) |
| 7 | `07_model_comparison.ipynb` | Robustness across model families |
| 8 | `08_shap_figures.ipynb` | Feature attribution (SHAP) |

## 3. Data Requirements

### 3.1 Sources

| | Flights | Weather |
|---|---|---|
| Source | BTS On-Time Performance (Kaggle "Flight Status Prediction", `Combined_Flights_<year>.parquet`) | NOAA Local Climatological Data (LCD), routine (`FM-15`) and special (`FM-16`) METAR reports |
| Period | January 2018 – July 2022 | 2018–2022 |
| Raw size | 29.2M records | Hourly grid of 6,135,358 station-hours (140 stations; 0.21% of hours without any report) |
| Granularity | One record per scheduled flight | One report per observation; aggregated to station-hour |
| Location in the project | `data/raw/bts/` | `data/raw/noaa_lcd/` (downloaded by step 1) |

### 3.2 Scope

- Ten busiest states: CA, TX, FL, IL, GA, NY, CO, NC, VA, WA; both origin and destination must lie in these states.
- Non-cancelled, non-diverted domestic flights with a valid label and scheduled elapsed time.
- Result: **9,981,331 flights across 137 airports**.
- Previous legs are searched in the complete BTS file covering all states (28.3M records after removing cancelled or diverted flights and records without a tail number); 90.4% of legs have a valid previous leg within 12 h.

### 3.3 Required fields and derived features

| Group | Fields / features |
|---|---|
| Flight fields read (20) | `FlightDate`, `Airline`, `Origin`, `Dest`, `Cancelled`, `Diverted`, `CRSDepTime`, `CRSArrTime`, `CRSElapsedTime`, `Distance`, `DistanceGroup`, `Month`, `DayOfWeek`, `DepTimeBlk`, `OriginState`, `DestState`, `Tail_Number`, `ArrDelay`, `ArrDelayMinutes`, `ArrDel15` |
| Schedule (16) | scheduled hours, distance, elapsed time, cyclic hour/day/month encodings, weekend, peak-hour and holiday flags |
| Identifiers | origin, destination, airline, tail number (out-of-fold target encoding, $m = 10$); state and departure time block (categorical) |
| Weather (31 per endpoint) | continuous METAR variables, weather-type flags (rain, snow, fog, mist, thunder, vicinity thunder, freezing, heavy), FAA flight category, missingness, 3-hour summaries |
| Propagation (6) | previous-leg status, departure delay, arrival delay, overdue minutes, scheduled turnaround, turnaround slack |

**Excluded (post-departure, leakage):** `DepTime`, `DepDelay`, `TaxiOut`, `WheelsOff`, `WheelsOn`, `TaxiIn`, `ArrTime`, `ActualElapsedTime`, `AirTime`. These fields of the predicted flight are never read into the modelling table; realized delays of the *previous* leg enter only through the horizon-aware propagation features, and only if realized before $t^0$. The signed `ArrDelay` is kept only to describe the target.

### 3.4 Data quality requirements

| Issue in a common BTS–NOAA join | Requirement implemented |
|---|---|
| Overnight arrivals assigned to the departure date (2.63% of flights misaligned by 24 h) | $t^{\text{arr}} = t^{\text{dep}} + $ `CRSElapsedTime` in UTC; 99.998% agreement with `CRSArrTime` |
| State-level time offsets (ECP, PNS, VPS, ELP misaligned by 1 h) | Per-airport IANA time zones |
| Precipitation summed over `FM-15` and `FM-16` (≈2.2× inflation) | Routine (`FM-15`) reports only |
| Substring parsing of weather codes; obscured-sky (`VV`) ceilings ignored | Token-level METAR parsing; `VV` treated as a ceiling; FAA flight category from ceiling and visibility |
| Gaps in hourly reports | Forward fill for at most 3 h; no backward filling |

### 3.5 Partitions

| Partition | Period | Flights |
|---|---|---|
| Train | 2018-01-01 – 2021-12-31 | 8,550,324 |
| Validation | 2022-01-01 – 2022-04-30 | 792,335 |
| Test | 2022-05-01 – 2022-07-31 | 638,672 |

All exploratory analyses use the training period unless stated otherwise.